In [14]:
import json
import pandas as pd
import glob
import requests

def merge_all_data() :

    batch_files = glob.glob("../data/raw/D1/Movies_Api_Respond_p*.json")

    merged_data = []

    for file_path in batch_files:
        with open(file_path, 'r') as file:
            batch_data = json.load(file)
            # tab = pd.DataFrame(batch_data['results'])
            merged_data.extend(batch_data['results'])


    with open("../data/processed/All_Movies.json", "w") as file:
        json.dump(merged_data, file, indent=4)

    return merged_data

df = merge_all_data()
df = pd.DataFrame(df)

print(df.columns.tolist())

['adult', 'backdrop_path', 'genre_ids', 'id', 'title', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'release_date', 'softcore', 'video', 'vote_average', 'vote_count']


In [ ]:
import json
import requests
from dotenv import load_dotenv
import os
import time


def get_api_key():
    load_dotenv()
    return os.getenv("TMDB_API_KEY")


def get_movie_details(id):
    API_KEY = get_api_key()

    params = {
        "api_key": API_KEY,
        "language": "en-US",
    }

    url = f"https://api.themoviedb.org/3/movie/{id}"

    data = None

    try:
        response = requests.get(url, params=params)

        if response.status_code == 200:
            print("Details success!")
            data = response.json()
        else:
            print("Details error:", response.status_code)
            print(response.text)

    except requests.exceptions.RequestException as e:
        print("Request failed:", e)

    return data


def get_movie_keywords(id):
    API_KEY = get_api_key()

    params = {
        "api_key": API_KEY,
        "language": "en-US",
    }

    url = f"https://api.themoviedb.org/3/movie/{id}/keywords"

    data = None

    try:
        response = requests.get(url, params=params)

        if response.status_code == 200:
            print("Keywords success!")
            data = response.json()
        else:
            print("Keywords error:", response.status_code)
            print(response.text)

    except requests.exceptions.RequestException as e:
        print("Request failed:", e)

    return data


def get_response_from_tmdb():

    with open("../data/processed/All_Movies.json", "r") as file:
        All_Movies = json.load(file)

    All_Movies_List = []

    for movie in All_Movies:

        movie_id = movie["id"]

        details = get_movie_details(movie_id)
        keywords_data = get_movie_keywords(movie_id)


        if details is None or keywords_data is None:
            continue

        genres = [
            genre["name"]
            for genre in details["genres"]
        ]


        keywords = [
            keyword["name"]
            for keyword in keywords_data["keywords"]
        ]


        movie_data = {
            "movie_id": movie["id"],
            "title": movie["title"],
            "overview": movie["overview"],
            "release_date": movie["release_date"],
            "runtime": details["runtime"],
            "original_language": movie["original_language"],
            "genres": genres,
            "keywords": keywords,
            "budget": details["budget"],
            "revenue": details["revenue"],
            "popularity": movie["popularity"],
            "vote_average": movie["vote_average"],
            "vote_count": movie["vote_count"]
        }


        All_Movies_List.append(movie_data)

        time.sleep(0.2)

    return All_Movies_List


All_Movies_List = get_response_from_tmdb()


with open("../data/raw/D2/Clean_Movies_test.json", "w") as f:
    json.dump(All_Movies_List, f, indent=4)